# 🇻🇳 Phân loại tin tức VietNamNet — SVC Linear Kernel

**Bài toán**: Phân loại tự động bài báo tiếng Việt vào **19 chủ đề**.

| Bước | Nội dung |
|------|----------|
| **1. Load data** | Đọc 19 file parquet từ thư mục Dataset |
| **2. Khám phá dữ liệu** | Phân bố class, độ dài văn bản, bảng tổng hợp |
| **3. Tiền xử lý** | ViTokenizer + loại stopwords (joblib parallel) |
| **4. TF-IDF** | Vectorize văn bản đã tokenize |
| **5. Huấn luyện** | SVC C=1, class_weight=balanced |
| **6. Đánh giá** | Classification report, confusion matrix, F1 per class |
| **7. Export** | Đóng gói pipeline suy luận |
| **8. Chẩn đoán** | In ra các tín hiệu cần xem để cải thiện model |

> **Mô hình**: `SVC(C=1, class_weight='balanced', probability=False)` + `TfidfVectorizer(max_features=150 000, ngram_range=(1,2), min_df=2, sublinear_tf=True)`
>
> **Caching**: mỗi bước nặng (tokenize / TF-IDF / train) tự kiểm tra cache pkl — nếu đã có sẽ bỏ qua tính toán, load trực tiếp.

---
## ⚙️ Section 0 — Setup
Chạy **mỗi lần** mở notebook.

In [ ]:
# ── Kiểm tra thư viện ───────────────────────────────────────────────────────
import importlib, sys

_REQUIRED = {
    "pandas":     "pandas",
    "numpy":      "numpy",
    "matplotlib": "matplotlib",
    "seaborn":    "seaborn",
    "sklearn":    "scikit-learn",
    "pyvi":       "pyvi",
    "joblib":     "joblib",
    "pyarrow":    "pyarrow",
    "tqdm":       "tqdm",
}

_missing = {pkg for mod, pkg in _REQUIRED.items() if importlib.util.find_spec(mod) is None}

if _missing:
    print("=" * 60)
    print("  KHONG THE TIEP TUC -- Thieu thu vien")
    print("=" * 60)
    print("  Cac goi chua duoc cai:")
    for _p in sorted(_missing):
        print(f"    - {_p}")
    print()
    _pip = " ".join(sorted(_missing))
    print("  Chay lenh sau roi restart kernel:")
    print(f"     pip install {_pip}")
    print("=" * 60)
    raise SystemExit("Thieu thu vien -- xem huong dan o tren.")

# ── Import ───────────────────────────────────────────────────────────────────
import os, re, pickle, time, datetime, warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print("Tat ca thu vien san sang --", sys.version.split()[0])

In [ ]:
# ── Đường dẫn ────────────────────────────────────────────────────────────────
NOTEBOOK_DIR        = os.getcwd()
DATASET_FOLDER      = os.path.normpath(os.path.join(NOTEBOOK_DIR, "..", "Dataset"))
STOPWORDS_FILE      = os.path.join(NOTEBOOK_DIR, "vietnamese-stopwords.txt")

TEMP_DIR            = os.path.join(NOTEBOOK_DIR, "temp")     # cache trung gian
RESULTS_DIR         = os.path.join(NOTEBOOK_DIR, "results")  # ảnh + report txt
MODEL_DIR           = os.path.join(NOTEBOOK_DIR, "model")    # model dùng cho app

for _d in [TEMP_DIR, RESULTS_DIR, MODEL_DIR]:
    os.makedirs(_d, exist_ok=True)

# temp: cache xử lý nặng (tokenize, tfidf)
PROCESSED_DATA_PATH = os.path.join(TEMP_DIR,  "processed_data.pkl")
TFIDF_DATA_PATH     = os.path.join(TEMP_DIR,  "tfidf_data.pkl")

# model: kết quả training + pipeline đóng gói cho app
MODEL_RESULTS_PATH  = os.path.join(MODEL_DIR, "model_results.pkl")
PIPELINE_PATH       = os.path.join(MODEL_DIR, "inference_pipeline.pkl")

# ── Tham số mô hình (đã chốt) ─────────────────────────────────────────────────
MAX_FEATURES = 150_000
NGRAM_RANGE  = (1, 2)
MIN_DF       = 2
C_VALUE      = 1
TEST_SIZE    = 0.15
RANDOM_STATE = 42

# ── Label map — 19 chủ đề ─────────────────────────────────────────────────────
LABEL_MAP = {
    "ban-doc":               "Bạn đọc",
    "bao-ve-nguoi-tieu-dung":"Bảo vệ người tiêu dùng",
    "bat-dong-san":          "Bất động sản",
    "chinh-tri":             "Chính trị",
    "cong-nghe":             "Công nghệ",
    "dan-toc-ton-giao":      "Dân tộc - Tôn giáo",
    "doi-song":              "Đời sống",
    "du-lich":               "Du lịch",
    "giao-duc":              "Giáo dục",
    "kinh-doanh":            "Kinh doanh",
    "oto-xe-may":            "Ô tô - Xe máy",
    "phap-luat":             "Pháp luật",
    "suc-khoe":              "Sức khỏe",
    "the-gioi":              "Thế giới",
    "the-thao":              "Thể thao",
    "thi-truong-tieu-dung":  "Thị trường tiêu dùng",
    "thoi-su":               "Thời sự",
    "tuan-viet-nam":         "Tuần Việt Nam",
    "van-hoa-giai-tri":      "Văn hóa - Giải trí",
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️", "OK": "✅", "WARN": "⚠️", "SAVE": "💾"}
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {icons.get(level,' ')} {msg}")

class timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *_): log(f"{self.label} — {time.time()-self.t:.1f}s", "OK")

def save_fig(fig, filename):
    path = os.path.join(RESULTS_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    log(f"Saved → {path}", "SAVE")
    plt.show(); plt.close(fig)

# ── Kiểm tra ─────────────────────────────────────────────────────────────────
_pq = [f for f in os.listdir(DATASET_FOLDER) if f.endswith(".parquet")] \
      if os.path.exists(DATASET_FOLDER) else []
print(f"✅ Config OK  |  {len(LABEL_MAP)} chủ đề  |  {len(_pq)} parquet files")
print(f"\n   Mô hình : SVC  C={C_VALUE}  class_weight=balanced  probability=False")
print(f"   TF-IDF  : max_features={MAX_FEATURES:,}  ngram={NGRAM_RANGE}  min_df={MIN_DF}")
print(f"\n   Cache:")
for _name, _path in [("temp/processed_data.pkl",      PROCESSED_DATA_PATH),
                      ("temp/tfidf_data.pkl",           TFIDF_DATA_PATH),
                      ("model/model_results.pkl",       MODEL_RESULTS_PATH),
                      ("model/inference_pipeline.pkl",  PIPELINE_PATH)]:
    print(f"     {_name:<30}: {'✅' if os.path.exists(_path) else '❌ chưa có'}")

In [ ]:
# ── Kiểm tra Dataset trước khi chạy ─────────────────────────────────────────
import pyarrow.parquet as _pq_check

_ok = True
_errors = []

# 1. Thư mục Dataset tồn tại
if not os.path.isdir(DATASET_FOLDER):
    _errors.append(f"❌ Không tìm thấy thư mục Dataset: {DATASET_FOLDER}")
    _ok = False
else:
    # 2. Số file parquet phải đúng bằng LABEL_MAP
    _pq_files = sorted(f for f in os.listdir(DATASET_FOLDER) if f.endswith('.parquet'))
    _expected = set(LABEL_MAP.keys())
    _found    = {f.replace('.parquet','') for f in _pq_files}
    _missing  = _expected - _found
    _extra    = _found - _expected

    if _missing:
        _errors.append(f"❌ Thiếu {len(_missing)} file parquet: {sorted(_missing)}")
        _ok = False
    if len(_found) != len(_expected):
        _errors.append(f"❌ Cần {len(_expected)} file, tìm thấy {len(_found)} file")
        _ok = False

    # 3. Từng file phải có dữ liệu (không rỗng)
    if _ok:
        for _f in _pq_files:
            _path = os.path.join(DATASET_FOLDER, _f)
            try:
                _meta = _pq_check.read_metadata(_path)
                if _meta.num_rows == 0:
                    _errors.append(f"❌ File rỗng (0 dòng): {_f}")
                    _ok = False
            except Exception as _e:
                _errors.append(f"❌ Không đọc được {_f}: {_e}")
                _ok = False

# 4. File stopwords tồn tại
if not os.path.isfile(STOPWORDS_FILE):
    _errors.append(f"❌ Không tìm thấy file stopwords: {STOPWORDS_FILE}")
    _ok = False

# Kết quả
if _ok:
    print(f"✅ Dataset OK — {len(_pq_files)} file parquet, tất cả có dữ liệu")
    print(f"✅ Stopwords OK — {STOPWORDS_FILE}")
else:
    print("\n" + "="*60)
    print("  KHÔNG THỂ TIẾP TỤC — Dataset chưa sẵn sàng")
    print("="*60)
    for _e in _errors:
        print(f"  {_e}")
    print("\n  👉 Chạy Crawling Data/crawl_data.ipynb để tạo Dataset trước.")
    print("="*60 + "\n")
    raise SystemExit("Dataset chưa sẵn sàng — xem hướng dẫn ở trên.")

---
## 📂 Section 1 — Load Dữ Liệu Thô

Đọc toàn bộ **19 file parquet** từ thư mục Dataset.

In [ ]:
# ── 1.1 Load raw data ────────────────────────────────────────────────────────
log(f"Đọc {len(LABEL_MAP)} file parquet từ {DATASET_FOLDER}...")
_records = []
for _fname in sorted(os.listdir(DATASET_FOLDER)):
    if not _fname.endswith(".parquet"): continue
    _lbl = LABEL_MAP.get(_fname.replace(".parquet", ""))
    if _lbl is None: continue
    _dfc = pd.read_parquet(os.path.join(DATASET_FOLDER, _fname))
    _dfc["label"] = _lbl
    log(f"  {_fname:<48}  {len(_dfc):>7,} bài  [{_lbl}]")
    _records.append(_dfc)

df_raw = pd.concat(_records, ignore_index=True)
log(f"Tổng raw: {len(df_raw):,} bài | {df_raw['label'].nunique()} chủ đề", "OK")

# ── 1.2 Kiểm tra & loại bỏ bài thiếu cả title lẫn content ───────────────────
_miss_title   = df_raw["title"].isna()   | (df_raw["title"].astype(str).str.strip()   == "")
_miss_content = df_raw["content"].isna() | (df_raw["content"].astype(str).str.strip() == "")
_miss_both    = _miss_title & _miss_content   # chỉ loại khi thiếu CẢ HAI

# Báo cáo tổng quan chất lượng dữ liệu
print(f"\n   [Kiểm tra chất lượng dữ liệu]")
print(f"   Thiếu title           : {_miss_title.sum():,} bài")
print(f"   Thiếu content         : {_miss_content.sum():,} bài")
print(f"   Thiếu cả 2 (→ loại)  : {_miss_both.sum():,} bài")

if _miss_both.any():
    _df_miss = df_raw[_miss_both]
    print(f"\n   Phân bổ {_miss_both.sum():,} bài bị loại theo chủ đề:\n")
    print(f"   {'Chủ đề':<38}  {'Số bài bị loại':>14}")
    print(f"   {'─'*55}")
    for _cls, _cnt in _df_miss["label"].value_counts().sort_index().items():
        print(f"   {_cls:<38}  {_cnt:>14,}")
    df_raw = df_raw[~_miss_both].reset_index(drop=True)
    log(f"Sau khi loại: {len(df_raw):,} bài còn lại", "OK")
else:
    print(f"\n✅ Không có bài nào thiếu cả title lẫn content.")

# ── 1.3 Chuẩn hoá kiểu dữ liệu ───────────────────────────────────────────────
df_raw["title"]    = df_raw["title"].fillna("").astype(str).str.strip()
df_raw["content"]  = df_raw["content"].fillna("").astype(str).str.strip()
df_raw["text_len"] = (df_raw["title"] + " " + df_raw["content"]).str.split().str.len()

print(f"\n   Cột dữ liệu : {list(df_raw.columns)}")
print(f"   Tổng cuối   : {len(df_raw):,} bài")

---
## 📊 Section 2 — Khám Phá Dữ Liệu (EDA)

Phân tích phân bố class, độ dài văn bản, thống kê tổng hợp.

In [ ]:
# ── 2.1 Phân bố class ────────────────────────────────────────────────────────
_vc    = df_raw["label"].value_counts().sort_values(ascending=True)
_total = len(df_raw)
_cmap  = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(_vc)))

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Horizontal bar
_bars = axes[0].barh(_vc.index, _vc.values, color=_cmap, edgecolor="white")
for bar, n in zip(_bars, _vc.values):
    axes[0].text(bar.get_width() + _total * 0.003,
                 bar.get_y() + bar.get_height() / 2,
                 f"{n:,}  ({n/_total*100:.1f}%)", va="center", fontsize=8.5)
axes[0].set_xlim(0, _vc.max() * 1.32)
axes[0].axvline(_vc.mean(), color="red", ls="--", alpha=0.8, label=f"Mean = {_vc.mean():,.0f}")
axes[0].set_xlabel("Số bài báo", fontsize=11)
axes[0].set_title("Số bài theo chủ đề", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=9); axes[0].grid(axis="x", alpha=0.3)

# Pie
axes[1].pie(_vc.values[::-1], labels=_vc.index[::-1],
            autopct="%1.1f%%", startangle=140, textprops={"fontsize": 8})
axes[1].set_title("Tỷ lệ %", fontsize=12, fontweight="bold")

_ir = _vc.max() / _vc.min()
fig.suptitle(f"Dataset: {_total:,} bài báo  |  {len(_vc)} chủ đề  |  Imbalance: {_ir:.1f}×",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save_fig(fig, "01_class_distribution.png")

print(f"\n{'─'*50}")
print(f"  Nhiều nhất : {_vc.idxmax():<30} {_vc.max():>8,}")
print(f"  Ít nhất    : {_vc.idxmin():<30} {_vc.min():>8,}")
print(f"  Imbalance  : {_ir:.1f}×  → dùng class_weight='balanced'")

In [ ]:
# ── 2.2 Phân bố độ dài văn bản ──────────────────────────────────────────────
_lens = df_raw["text_len"]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Histogram toàn bộ
axes[0].hist(_lens, bins=80, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(_lens.mean(),   color="red",    ls="--", lw=1.5,
                label=f"Mean = {_lens.mean():.0f}")
axes[0].axvline(_lens.median(), color="orange", ls="--", lw=1.5,
                label=f"Median = {_lens.median():.0f}")
axes[0].set_xlabel("Số từ (title + content)")
axes[0].set_ylabel("Số bài")
axes[0].set_title("Phân bố độ dài (toàn bộ)", fontweight="bold")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Histogram cắt P95
_p95 = _lens.quantile(0.95)
axes[1].hist(_lens[_lens <= _p95], bins=80, color="teal", edgecolor="white", alpha=0.85)
axes[1].set_xlabel("Số từ")
axes[1].set_title(f"Phân bố (≤ P95 = {_p95:.0f} từ)", fontweight="bold")
axes[1].grid(alpha=0.3)

# Boxplot per class
_cls_ord = list(df_raw["label"].value_counts().sort_values().index)
_lbc = [df_raw[df_raw["label"] == c]["text_len"].values for c in _cls_ord]
bp = axes[2].boxplot(_lbc, vert=False, patch_artist=True,
                     medianprops=dict(color="red", linewidth=2))
for patch, color in zip(bp["boxes"], plt.cm.Set3(np.linspace(0, 1, len(_cls_ord)))):
    patch.set_facecolor(color)
axes[2].set_yticks(range(1, len(_cls_ord) + 1))
axes[2].set_yticklabels(_cls_ord, fontsize=8)
axes[2].set_xlabel("Số từ")
axes[2].set_title("Boxplot độ dài theo chủ đề", fontweight="bold")
axes[2].grid(axis="x", alpha=0.3)

fig.suptitle("Phân tích độ dài văn bản (dữ liệu thô)", fontsize=13, fontweight="bold")
fig.tight_layout()
save_fig(fig, "02_text_length.png")

print(f"\n{'─'*40}")
for _lbl, _fn in [("Min", "min"), ("Max", "max"), ("Mean", "mean"),
                   ("Median", "median"), ("P90", None), ("P95", None)]:
    _v = _lens.quantile(0.9) if _lbl == "P90" else \
         _lens.quantile(0.95) if _lbl == "P95" else getattr(_lens, _fn)()
    print(f"  {_lbl:<8}: {_v:>8,.1f} từ") 

In [ ]:
# ── 2.3 Bảng tổng hợp dataset ───────────────────────────────────────────────
_rows = []
for _cls in sorted(df_raw["label"].unique()):
    _sub = df_raw[df_raw["label"] == _cls]["text_len"]
    _rows.append({
        "Chủ đề":      _cls,
        "Số bài":      len(_sub),
        "Tỷ lệ %":     round(len(_sub) / len(df_raw) * 100, 2),
        "TB từ":       round(_sub.mean(), 0),
        "Median từ":   round(_sub.median(), 0),
        "Min từ":      int(_sub.min()),
        "Max từ":      int(_sub.max()),
    })
_df_sum = (pd.DataFrame(_rows)
             .sort_values("Số bài", ascending=False)
             .reset_index(drop=True))
_df_sum.index += 1

print("📊 BẢNG TỔNG HỢP DỮ LIỆU\n")
display(
    _df_sum.style
    .background_gradient(subset=["Số bài"], cmap="Blues")
    .background_gradient(subset=["TB từ"], cmap="Greens")
    .format({"Số bài": "{:,}", "Tỷ lệ %": "{:.2f}%", "TB từ": "{:.0f}", "Median từ": "{:.0f}"})
) 

---
## 🔤 Section 3 — Tiền Xử Lý Văn Bản

**Các bước**: làm sạch → ViTokenizer (song song) → loại stopwords → lưu cache.

- Cell 3.1: tokenize toàn bộ corpus, lưu `processed_data.pkl` *(nặng, ~20–40 phút; bỏ qua nếu cache đã có)*
- Cell 3.2: load dữ liệu đã xử lý vào bộ nhớ *(luôn chạy)*

In [ ]:
# ── 3.1 Tokenize + Lưu cache ─────────────────────────────────────────────────
# Load stopwords LUÔN thực hiện — cần cho cả training lẫn inference
with open(STOPWORDS_FILE, encoding="utf-8") as _f:
    STOPWORDS = frozenset(line.strip() for line in _f if line.strip())
log(f"Stopwords loaded: {len(STOPWORDS):,}", "OK")

if os.path.exists(PROCESSED_DATA_PATH):
    print(f"✅ Cache tồn tại: {PROCESSED_DATA_PATH}  — bỏ qua tokenization")
else:
    log("Bắt đầu tiền xử lý văn bản...")

    # Ghép title (×3) + content để tăng trọng số tiêu đề
    df_proc = df_raw[["label"]].copy()
    df_proc["full_text"] = (df_raw["title"] + " " + df_raw["title"]
                            + " " + df_raw["title"] + " " + df_raw["content"])
    df_proc = df_proc[df_proc["full_text"].str.strip() != ""].reset_index(drop=True)
    log(f"Văn bản hợp lệ: {len(df_proc):,}", "OK")

    # Hàm tokenize một văn bản (import bên trong vì chạy trong subprocess riêng)
    def _clean_one(text, sw_list):
        from pyvi import ViTokenizer
        import re
        sw = set(sw_list)
        if not isinstance(text, str): return ""
        text = text.lower()
        text = re.sub(r"[^\w\s]", " ", text)
        text = re.sub(r"\d+", " ", text)
        text = ViTokenizer.tokenize(text)
        return re.sub(r"\s+", " ",
                      " ".join(t for t in text.split() if t not in sw)).strip()

    from joblib import Parallel, delayed
    from multiprocessing import cpu_count
    _sw_list = list(STOPWORDS)
    log(f"Parallel tokenize {len(df_proc):,} bài | {cpu_count()} CPU cores...")
    df_proc["clean_text"] = Parallel(n_jobs=min(cpu_count() // 2, 8), backend="loky", verbose=1, pre_dispatch="2*n_jobs")(
        delayed(_clean_one)(t, _sw_list) for t in df_proc["full_text"].tolist()
    )
    df_proc = df_proc[df_proc["clean_text"].str.strip() != ""].reset_index(drop=True)
    log(f"Sau tokenization: {len(df_proc):,} bài", "OK")

    _classes  = sorted(df_proc["label"].unique().tolist())
    _label2id = {l: i for i, l in enumerate(_classes)}
    _id2label = {i: l for l, i in _label2id.items()}
    df_proc["label_id"] = df_proc["label"].map(_label2id)

    with timer("Lưu processed_data.pkl"):
        with open(PROCESSED_DATA_PATH, "wb") as _f:
            pickle.dump({
                "df":        df_proc,
                "label2id":  _label2id,
                "id2label":  _id2label,
                "stopwords": sorted(STOPWORDS),   # lưu để dùng khi export pipeline
            }, _f)
    log(f"Saved ({os.path.getsize(PROCESSED_DATA_PATH)/1e6:.1f} MB)", "SAVE")

In [ ]:
# ── 3.2 Load processed data (luôn chạy) ──────────────────────────────────────
with timer("Load processed_data.pkl"):
    with open(PROCESSED_DATA_PATH, "rb") as f:
        _s = pickle.load(f)

df       = _s["df"]
label2id = _s["label2id"]
id2label = _s["id2label"]
classes  = sorted(df["label"].unique().tolist())

# Đảm bảo STOPWORDS luôn có trong scope (phòng trường hợp bỏ qua Cell 3.1)
if "STOPWORDS" not in dir():
    STOPWORDS = frozenset(_s.get("stopwords", []))

_vc3 = df["label"].value_counts()
print(f"✅ Processed data: {len(df):,} bài | {len(classes)} chủ đề\n")
print(f"   {'Chủ đề':<38}  {'Bài':>8}  {'%':>5}  TB tokens")
print(f"   {'─'*60}")
for _cls in classes:
    _n   = _vc3.get(_cls, 0)
    _tlen = df[df["label"] == _cls]["clean_text"].str.split().str.len().mean()
    print(f"   {_cls:<38}  {_n:>8,}  {_n/len(df)*100:>4.1f}%  {_tlen:>6.0f}")

---
## 🔢 Section 4 — Trích Xuất Đặc Trưng (TF-IDF)

`TfidfVectorizer(max_features=150 000, ngram_range=(1,2), min_df=2, sublinear_tf=True)`

In [ ]:
# ── 4.1 Fit TF-IDF + Split train/test + Lưu cache ────────────────────────────
if os.path.exists(TFIDF_DATA_PATH):
    print(f"✅ Cache tồn tại: {TFIDF_DATA_PATH}")
    print("   → Bỏ qua TF-IDF, tự động load ở Cell 4.2")
else:
    log(f"Train/test split ({int(TEST_SIZE*100)}% test, stratified)...")
    X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
        df["clean_text"], df["label"],
        test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label"]
    )
    log(f"Train: {len(X_tr_raw):,}  |  Test: {len(X_te_raw):,}", "OK")

    log(f"Fit TF-IDF — max_features={MAX_FEATURES:,}  ngram={NGRAM_RANGE}  min_df={MIN_DF}...")
    _vec = TfidfVectorizer(
        max_features=MAX_FEATURES,
        ngram_range=NGRAM_RANGE,
        min_df=MIN_DF,
        sublinear_tf=True,
        analyzer="word",
    )
    with timer("fit_transform (train)"):
        X_train = _vec.fit_transform(X_tr_raw)
    with timer("transform (test)"):
        X_test  = _vec.transform(X_te_raw)

    log(f"X_train: {X_train.shape}  |  vocab: {len(_vec.vocabulary_):,}", "OK")

    with timer("Lưu tfidf_data.pkl"):
        with open(TFIDF_DATA_PATH, "wb") as f:
            pickle.dump({
                "vectorizer": _vec,
                "X_train": X_train, "X_test": X_test,
                "y_train": y_tr,    "y_test": y_te,
            }, f)
    log(f"Saved ({os.path.getsize(TFIDF_DATA_PATH)/1e6:.1f} MB)", "SAVE") 

In [ ]:
# ── 4.2 Load TF-IDF data (luôn chạy) ────────────────────────────────────────
with timer("Load tfidf_data.pkl"):
    with open(TFIDF_DATA_PATH, "rb") as f:
        _t = pickle.load(f)

vectorizer = _t["vectorizer"]
X_train    = _t["X_train"]
X_test     = _t["X_test"]
y_train    = _t["y_train"]
y_test     = _t["y_test"]

_sp = 1.0 - X_train.nnz / (X_train.shape[0] * X_train.shape[1])
print(f"✅ TF-IDF data loaded:")
print(f"   X_train  : {X_train.shape}  ({X_train.nnz:,} non-zero)")
print(f"   X_test   : {X_test.shape}")
print(f"   Vocab    : {X_train.shape[1]:,} features")
print(f"   Sparsity : {_sp:.4%}")
print(f"   Train    : {len(y_train):,}  |  Test: {len(y_test):,}") 

In [ ]:
# ── 4.3 Phân tích TF-IDF Vocabulary ─────────────────────────────────────────
_feat     = vectorizer.get_feature_names_out()
_idf_ser  = pd.Series(vectorizer.idf_, index=_feat)
_n_uni    = sum(1 for t in _feat if "_" not in t)
_n_bi     = len(_feat) - _n_uni
_top_com  = _idf_ser.nsmallest(20)
_top_rare = _idf_ser.nlargest(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(_top_com.index[::-1],  _top_com.values[::-1],  color="steelblue", edgecolor="white")
axes[0].set_title("Top 20 từ phổ biến nhất (IDF thấp)", fontweight="bold")
axes[0].set_xlabel("IDF"); axes[0].grid(axis="x", alpha=0.3)

axes[1].barh(_top_rare.index[::-1], _top_rare.values[::-1], color="coral", edgecolor="white")
axes[1].set_title("Top 20 từ đặc trưng nhất (IDF cao)", fontweight="bold")
axes[1].set_xlabel("IDF"); axes[1].grid(axis="x", alpha=0.3)

fig.suptitle(
    f"TF-IDF Vocab: {len(_feat):,} features  |  Unigram: {_n_uni:,}  |  Bigram: {_n_bi:,}",
    fontsize=12, fontweight="bold")
fig.tight_layout()
save_fig(fig, "03_tfidf_vocab.png")

print(f"\n   Vocab size  : {len(_feat):,}")
print(f"   Unigram     : {_n_uni:,}  ({_n_uni/len(_feat):.1%})")
print(f"   Bigram      : {_n_bi:,}  ({_n_bi/len(_feat):.1%})")
print(f"   IDF range   : {vectorizer.idf_.min():.3f} → {vectorizer.idf_.max():.3f}") 

---
## 🤖 Section 5 — Huấn Luyện Mô Hình

**SVC** — mô hình SVM dùng cấu hình mặc định của `sklearn.svm.SVC` cho phân loại văn bản trên ma trận TF-IDF.

| Tham số | Giá trị | Lý do |
|---------|---------|-------|
| `C` | 1 | Cân bằng regularization / fit |
| `class_weight` | balanced | Tự động điều chỉnh trọng số theo tỷ lệ class trong tập train |
| `probability` | False | App sẽ suy ra confidence từ `decision_function` |

In [ ]:
# -- 5.1 Train + Luu cache ---------------------------------------------------
MODEL_NAME = "SVC"

if os.path.exists(MODEL_RESULTS_PATH):
    log("Cache ket qua ton tai, loading...", "OK")
    with open(MODEL_RESULTS_PATH, "rb") as f:
        _res = pickle.load(f)
else:
    log(f"Training {MODEL_NAME} (C={C_VALUE}, probability=False)...")
    _model = SVC(
        C=C_VALUE,
        class_weight="balanced",
        probability=False,
        random_state=RANDOM_STATE,
    )

    _t0       = time.time()
    _ts_start = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{_ts_start}] Bat dau train ({X_train.shape[0]:,} mau)...", flush=True)
    _model.fit(X_train, y_train)

    _train_dur = time.time() - _t0
    _th, _tr   = divmod(int(_train_dur), 3600)
    _tm, _ts   = divmod(_tr, 60)
    _dur_str   = f"{_th}h {_tm:02d}m {_ts:02d}s" if _th else f"{_tm}m {_ts:02d}s"
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Train xong! {_dur_str}", flush=True)

    log("Predict tren tap test...")
    _t1     = time.time()
    _y_pred = _model.predict(X_test)
    log(f"Predict xong! ({time.time()-_t1:.1f}s)", "OK")

    _acc = accuracy_score(y_test, _y_pred)
    _f1w = f1_score(y_test, _y_pred, average="weighted")
    _f1m = f1_score(y_test, _y_pred, average="macro")
    _f1p = f1_score(y_test, _y_pred, labels=classes, average=None)

    _res = {
        "model":        _model,
        "y_pred":       _y_pred,
        "y_test":       y_test.values,
        "accuracy":     _acc,
        "f1_weighted":  _f1w,
        "f1_macro":     _f1m,
        "f1_per_class": dict(zip(classes, _f1p)),
        "classes":      classes,
        "train_time":   _train_dur,
        "config": {
            "model_name":   MODEL_NAME,
            "kernel":       "default",
            "probability":  False,
            "C":            C_VALUE,
            "max_features": MAX_FEATURES,
            "ngram_range":  NGRAM_RANGE,
            "min_df":       MIN_DF,
            "test_size":    TEST_SIZE,
            "n_train":      X_train.shape[0],
            "n_test":       X_test.shape[0],
        },
    }
    with open(MODEL_RESULTS_PATH, "wb") as f:
        pickle.dump(_res, f)
    log(f"Saved: {MODEL_RESULTS_PATH}", "SAVE")

# Expose sang scope toan cuc
y_pred       = _res["y_pred"]
y_test_arr   = _res["y_test"]
model_acc    = _res["accuracy"]
model_f1w    = _res["f1_weighted"]
model_f1m    = _res["f1_macro"]
f1_per_class = _res["f1_per_class"]
classes      = _res["classes"]
_train_time  = _res.get("train_time", 0)

_h, _rem = divmod(int(_train_time), 3600)
_m, _s   = divmod(_rem, 60)
_dur_disp = f"{_h}h {_m:02d}m {_s:02d}s" if _h else f"{_m}m {_s:02d}s" if _m else f"{_s}s"

SEP = "=" * 55
print(f"\n{SEP}")
print(f"  {MODEL_NAME} -- C={C_VALUE}  max_features={MAX_FEATURES:,}")
print(f"  class_weight : balanced")
print(f"  probability  : False")
print(f"  Thoi gian train : {_dur_disp}")
print(f"  {chr(45)*52}")
print(f"  Accuracy   : {model_acc:.4f}  ({model_acc*100:.2f}%)")
print(f"  F1-weighted: {model_f1w:.4f}")
print(f"  F1-macro   : {model_f1m:.4f}")
print(f"{SEP}")

---
## 📈 Section 6 — Đánh Giá Mô Hình

Classification report, confusion matrix, F1 từng chủ đề.

In [ ]:
# ── 6.1 Classification Report ────────────────────────────────────────────────
# Load nếu chưa có trong bộ nhớ
if "y_pred" not in dir():
    with open(MODEL_RESULTS_PATH, "rb") as f:
        _res = pickle.load(f)
    y_pred       = _res["y_pred"]; y_test_arr = _res["y_test"]
    model_acc    = _res["accuracy"]; model_f1w = _res["f1_weighted"]
    model_f1m    = _res["f1_macro"]; f1_per_class = _res["f1_per_class"]
    classes      = _res["classes"]

_report = classification_report(y_test_arr, y_pred, target_names=classes, digits=4)

print(f"{'='*65}")
print(f"  CLASSIFICATION REPORT")
print(f"  Mô hình : SVC  C={_res['config']['C']}"
      f"  max_features={_res['config']['max_features']:,}")
print(f"  Dữ liệu : Train {_res['config']['n_train']:,}  |  Test {_res['config']['n_test']:,}")
print(f"  class_weight : balanced")
print(f"  {'─'*62}")
print(f"  Accuracy   : {model_acc:.4f}")
print(f"  F1-weighted: {model_f1w:.4f}")
print(f"  F1-macro   : {model_f1m:.4f}")
print(f"{'='*65}")
print(_report)

_rp = os.path.join(RESULTS_DIR, "classification_report.txt")
with open(_rp, "w", encoding="utf-8") as f:
    f.write(f"SVC C={_res['config']['C']} "
            f"max_features={_res['config']['max_features']:,}\n")
    f.write(f"class_weight: balanced\n")
    f.write(f"Accuracy: {model_acc:.4f}\n")
    f.write(f"F1-weighted: {model_f1w:.4f}\n")
    f.write(f"F1-macro: {model_f1m:.4f}\n\n")
    f.write(_report)
log(f"Saved: {_rp}", "SAVE")

In [ ]:
# ── 6.2 Confusion Matrix ─────────────────────────────────────────────────────
_cm  = confusion_matrix(y_test_arr, y_pred, labels=classes)
_cmn = _cm.astype(float) / _cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(28, 11))
for ax, data, fmt, cmap, title in [
    (axes[0], _cmn, ".2f", "Blues",   "Normalized"),
    (axes[1], _cm,  "d",   "Oranges", "Raw Counts"),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                xticklabels=classes, yticklabels=classes,
                linewidths=0.3, linecolor="white", ax=ax,
                annot_kws={"size": 7})
    ax.set_xlabel("Nhãn dự đoán", fontsize=10)
    ax.set_ylabel("Nhãn thật", fontsize=10)
    ax.set_title(f"Confusion Matrix ({title})", fontsize=12, fontweight="bold")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    plt.setp(ax.get_yticklabels(), rotation=0,  fontsize=8)

fig.suptitle(
    f"Confusion Matrix — SVC C={C_VALUE} | {len(classes)} chủ đề | "
    f"Accuracy={model_acc:.4f}",
    fontsize=13, fontweight="bold")
fig.tight_layout()
save_fig(fig, "04_confusion_matrix.png") 

In [ ]:
# ── 6.3 F1-score theo từng chủ đề ────────────────────────────────────────────
_f1_items = sorted(f1_per_class.items(), key=lambda x: x[1])
_cls_names = [x[0] for x in _f1_items]
_cls_f1    = [x[1] for x in _f1_items]
_bar_colors = ["#d73027" if v < 0.80 else
               "#fee090" if v < 0.90 else
               "#4dac26" for v in _cls_f1]

fig, ax = plt.subplots(figsize=(12, 9))
_bars = ax.barh(_cls_names, _cls_f1, color=_bar_colors, edgecolor="white", height=0.7)
for bar, v in zip(_bars, _cls_f1):
    ax.text(bar.get_width() + 0.004,
            bar.get_y() + bar.get_height() / 2,
            f"{v:.4f}", va="center", fontsize=9)

ax.axvline(model_f1m, color="navy",      ls="--", lw=2,
           label=f"F1-macro = {model_f1m:.4f}")
ax.axvline(model_f1w, color="darkgreen", ls="-.", lw=2,
           label=f"F1-weighted = {model_f1w:.4f}")
ax.set_xlim(0, 1.10)
ax.set_xlabel("F1-score", fontsize=11)
ax.set_title("F1-score theo từng chủ đề (sắp xếp tăng dần)",
             fontsize=13, fontweight="bold")
ax.grid(axis="x", alpha=0.3)

ax.legend(handles=[
    Patch(color="#d73027", label="F1 < 0.80"),
    Patch(color="#fee090", label="0.80 ≤ F1 < 0.90"),
    Patch(color="#4dac26", label="F1 ≥ 0.90"),
    plt.Line2D([0],[0], color="navy",      ls="--", lw=2,
               label=f"F1-macro = {model_f1m:.4f}"),
    plt.Line2D([0],[0], color="darkgreen", ls="-.", lw=2,
               label=f"F1-weighted = {model_f1w:.4f}"),
], fontsize=9, loc="lower right")

fig.tight_layout()
save_fig(fig, "05_f1_per_class.png") 

In [ ]:
# ── 6.4 Tóm tắt kết quả cuối cùng ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  📋 KẾT QUẢ MÔ HÌNH PHÂN LOẠI TIN TỨC VIETNAMNET")
print(f"{'='*60}")
print(f"\n  [CẤU HÌNH MÔ HÌNH]")
print(f"    Thuật toán   : SVC")
print(f"    C            : {_res['config']['C']}")
print(f"    class_weight : balanced")
print(f"    probability  : False")

print(f"\n  [CẤU HÌNH TF-IDF]")
print(f"    max_features : {_res['config']['max_features']:,}")
print(f"    ngram_range  : {_res['config']['ngram_range']}")
print(f"    min_df       : {_res['config']['min_df']}")
print(f"    sublinear_tf : True")

print(f"\n  [DỮ LIỆU]")
print(f"    Số chủ đề    : {len(classes)}")
print(f"    Train        : {_res['config']['n_train']:,} bài ({(1-TEST_SIZE)*100:.0f}%)")
print(f"    Test         : {_res['config']['n_test']:,} bài  ({TEST_SIZE*100:.0f}%)")

print(f"\n  [HIỆU NĂNG TRÊN TẬP TEST]")
print(f"    Accuracy     : {model_acc:.4f}  ({model_acc*100:.2f}%)")
print(f"    F1-weighted  : {model_f1w:.4f}")
print(f"    F1-macro     : {model_f1m:.4f}")

print(f"\n  [5 CHỦ ĐỀ KHÓ PHÂN LOẠI NHẤT]")
for _cls, _v in sorted(f1_per_class.items(), key=lambda x: x[1])[:5]:
    print(f"    {_cls:<38}  F1 = {_v:.4f}")

print(f"\n  [5 CHỦ ĐỀ DỄ PHÂN LOẠI NHẤT]")
for _cls, _v in sorted(f1_per_class.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"    {_cls:<38}  F1 = {_v:.4f}")

_cm2 = confusion_matrix(y_test_arr, y_pred, labels=classes)
_confused = sorted(
    [(_cm2[i, j], classes[i], classes[j])
     for i in range(len(classes)) for j in range(len(classes)) if i != j and _cm2[i,j] > 0],
    reverse=True
)
print(f"\n  [TOP 5 CẶP CHỦ ĐỀ BỊ NHẦM NHIỀU NHẤT]")
for _cnt, _tr, _pr in _confused[:5]:
    print(f"    {_tr:<28} → {_pr:<28}  {_cnt:>4} lần")

print(f"\n  [FILE KẾT QUẢ ĐÃ LƯU]")
for _f in sorted(os.listdir(RESULTS_DIR)):
    _fp = os.path.join(RESULTS_DIR, _f)
    print(f"    {_f:<35}  ({os.path.getsize(_fp)/1024:.1f} KB)")
print(f"\n{'='*60}")

---
## 📦 Section 7 — Export Inference Pipeline

Đóng gói toàn bộ pipeline suy luận vào **một file duy nhất** (`inference_pipeline.pkl`) để tái sử dụng cho inference.

File chứa: `vectorizer` + `model` + `stopwords` + `classes` + `config`.

> **Quan trọng**: khi inference, tiền xử lý văn bản đầu vào phải **khớp chính xác** với pipeline training:
> `(title + " " + title + " " + title + " " + content)` → lowercase → strip punct → strip digits → ViTokenizer → loại stopwords → `vectorizer.transform()`

In [ ]:
# ── 7.1 Export inference pipeline ────────────────────────────────────────────
_pipeline = {
    "vectorizer": vectorizer,
    "model":      _res["model"],
    "stopwords":  sorted(STOPWORDS),
    "classes":    classes,
    "config": {
        "title_weight":  3,             # title lặp 3 lần trước content
        "lowercase":     True,
        "remove_punct":  True,          # re.sub(r"[^\w\s]", " ")
        "remove_digits": True,          # re.sub(r"\d+", " ")
        "tokenizer":     "pyvi.ViTokenizer",
        "model_name":    MODEL_NAME,
        "kernel":        "default",
        "probability":   False,
        "max_features":  MAX_FEATURES,
        "ngram_range":   NGRAM_RANGE,
        "min_df":        MIN_DF,
        "sublinear_tf":  True,
        "C":             C_VALUE,
        "class_weight":  "balanced",
    },
}

with open(PIPELINE_PATH, "wb") as f:
    pickle.dump(_pipeline, f)
log(f"Saved: {PIPELINE_PATH}  ({os.path.getsize(PIPELINE_PATH)/1e6:.1f} MB)", "SAVE")

# ── Smoke test: load lại pipeline và predict 3 bài ngẫu nhiên ─────────────────
with open(PIPELINE_PATH, "rb") as f:
    _p = pickle.load(f)

def _predict_one(title, content, pipeline):
    from pyvi import ViTokenizer
    sw    = set(pipeline["stopwords"])
    text  = (str(title) + " " + str(title) + " " + str(title) + " " + str(content)).lower()
    text  = re.sub(r"[^\w\s]", " ", text)
    text  = re.sub(r"\d+",     " ", text)
    text  = ViTokenizer.tokenize(text)
    clean = re.sub(r"\s+", " ", " ".join(t for t in text.split() if t not in sw)).strip()
    vec   = pipeline["vectorizer"].transform([clean])
    return pipeline["model"].predict(vec)[0]

print(f"\n  [SMOKE TEST — 3 bài ngẫu nhiên từ df_raw]\n")
_rng = np.random.default_rng(99)
_ok  = 0
for _i in _rng.choice(len(df_raw), size=3, replace=False):
    _row  = df_raw.iloc[_i]
    _pred = _predict_one(_row["title"], _row["content"], _p)
    _ok  += int(_pred == _row["label"])
    _mark = "✅" if _pred == _row["label"] else "❌"
    print(f"  {_mark}  Thật: {_row['label']:<32}  Dự đoán: {_pred}")
print(f"\n  Kết quả smoke test: {_ok}/3")


---
## Section 8 - Diagnostic Print

Section nay chi print ra cac tin hieu can xem de cai thien model, khong train them.


In [ ]:
# -- 8.1 Per-class metric table ---------------------------------------------
if "y_pred" not in dir():
    with open(MODEL_RESULTS_PATH, "rb") as f:
        _res = pickle.load(f)
    y_pred       = _res["y_pred"]; y_test_arr = _res["y_test"]
    model_acc    = _res["accuracy"]; model_f1w = _res["f1_weighted"]
    model_f1m    = _res["f1_macro"]; f1_per_class = _res["f1_per_class"]
    classes      = _res["classes"]

_diag_report = classification_report(
    y_test_arr, y_pred, target_names=classes, digits=4,
    output_dict=True, zero_division=0
)
_diag_df = pd.DataFrame([
    {
        "class": cls,
        "precision": _diag_report[cls]["precision"],
        "recall":    _diag_report[cls]["recall"],
        "f1":        _diag_report[cls]["f1-score"],
        "support":   int(_diag_report[cls]["support"]),
    }
    for cls in classes
])
_diag_df["pr_gap"] = _diag_df["recall"] - _diag_df["precision"]
_diag_df["support_pct"] = _diag_df["support"] / max(int(_diag_df["support"].sum()), 1)

print(f"\n{'='*88}")
print("[SVM DIAGNOSTIC SUMMARY]")
print(f"  Accuracy    : {model_acc:.4f}")
print(f"  F1-weighted : {model_f1w:.4f}")
print(f"  F1-macro    : {model_f1m:.4f}")
print(f"  Gap (w-m)   : {model_f1w - model_f1m:+.4f}")
print(f"{'='*88}")

print("\n[5 WEAKEST CLASSES BY F1]")
for _, r in _diag_df.sort_values(["f1", "support"], ascending=[True, True]).head(5).iterrows():
    print(f"  {r['class']:<30}  F1={r['f1']:.4f}  P={r['precision']:.4f}  R={r['recall']:.4f}  support={int(r['support'])}")

print("\n[5 SMALLEST TEST CLASSES]")
for _, r in _diag_df.sort_values(["support", "f1"], ascending=[True, True]).head(5).iterrows():
    print(f"  {r['class']:<30}  support={int(r['support']):>5}  ({r['support_pct']*100:>5.2f}%)  F1={r['f1']:.4f}")

print("\n[CLASSES WITH LOW RECALL VS PRECISION]")
for _, r in _diag_df.sort_values("pr_gap", ascending=True).head(5).iterrows():
    print(f"  {r['class']:<30}  gap={r['pr_gap']:+.4f}  P={r['precision']:.4f}  R={r['recall']:.4f}")

print("\n[CLASSES WITH LOW PRECISION VS RECALL]")
for _, r in _diag_df.sort_values("pr_gap", ascending=False).head(5).iterrows():
    print(f"  {r['class']:<30}  gap={r['pr_gap']:+.4f}  P={r['precision']:.4f}  R={r['recall']:.4f}")

# -- 8.2 Top confusion pairs ------------------------------------------------
_cm_diag = confusion_matrix(y_test_arr, y_pred, labels=classes)
_pairs = []
for i, true_cls in enumerate(classes):
    _support = max(int(_cm_diag[i].sum()), 1)
    for j, pred_cls in enumerate(classes):
        if i == j or _cm_diag[i, j] == 0:
            continue
        _pairs.append({
            "count": int(_cm_diag[i, j]),
            "rate":  float(_cm_diag[i, j] / _support),
            "true":  true_cls,
            "pred":  pred_cls,
        })
_pairs_df = pd.DataFrame(_pairs).sort_values(["count", "rate"], ascending=[False, False]) if _pairs else pd.DataFrame(columns=["count", "rate", "true", "pred"])

print("\n[TOP 8 CONFUSION PAIRS]")
if len(_pairs_df) == 0:
    print("  No off-diagonal confusion pairs.")
else:
    for _, r in _pairs_df.head(8).iterrows():
        print(f"  {r['true']:<28} -> {r['pred']:<28}  {int(r['count']):>4} times  ({r['rate']*100:>5.2f}%)")

# -- 8.3 Next-step hints ----------------------------------------------------
print("\n[SVM NEXT-STEP HINTS]")
if (model_f1w - model_f1m) > 0.06:
    print("  - Weighted is much higher than macro -> class imbalance is still strong; inspect low-support classes first.")
else:
    print("  - Weighted vs macro gap is not too large -> focus on specific confusion pairs.")

_low_recall = _diag_df[_diag_df["pr_gap"] < -0.08].sort_values("f1").head(3)["class"].tolist()
_low_precision = _diag_df[_diag_df["pr_gap"] > 0.08].sort_values("f1").head(3)["class"].tolist()
_very_low_f1 = _diag_df[_diag_df["f1"] < 0.80].sort_values("f1")["class"].tolist()

if _low_recall:
    print(f"  - Low recall classes: {', '.join(_low_recall)}")
    print("    Try small class-weight boost, add more data, or expand bigram/max_features if vocabulary is weak.")
if _low_precision:
    print(f"  - Low precision classes: {', '.join(_low_precision)}")
    print("    Review over-predicted classes; consider reducing class weight or inspect confusion pairs.")
if _very_low_f1:
    print(f"  - Classes with F1 < 0.80: {', '.join(_very_low_f1[:6])}{' ...' if len(_very_low_f1) > 6 else ''}")
    print("    Prioritize adding data or improving title/content feature separation for this group.")

if "X_train" in dir():
    _density = float(X_train.nnz / (X_train.shape[0] * X_train.shape[1])) if X_train.shape[0] * X_train.shape[1] > 0 else 0.0
    print(f"  - TF-IDF train matrix: {X_train.shape[0]:,} x {X_train.shape[1]:,}  |  density = {_density:.6f}")
    print("    If training is too slow, reduce max_features or try feature selection / SVD before heavier models.")

print("  - If confusion is concentrated in a few pairs, inspect those pairs directly before global tuning.")
print("  - If SVM is already tuned but macro F1 is still low, keep it as baseline and push hard cases to PhoBERT / ensemble.")
